In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
import gradio as gr 

In [3]:
load_dotenv(override=True)

google_api_key = os.getenv('GOOGLE_API_KEY')

if google_api_key:
    print(f"Google API 키가 존재하며 {google_api_key[:8]}로 시작합니다")
else:
    print("Google API 키가 설정되어 있지 않습니다")

Google API 키가 존재하며 AQ.Ab8RN로 시작합니다


In [5]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [7]:
system_message = "당신은 도움이 되는 어시스턴트입니다"

def message_gemini(prompt):
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages)
    return response.choices[0].message.content

# 사용자 인터페이스

In [8]:

def shout(text):
    print(f"입력 {text}로 shout가 호출되었습니다")
    return text.upper()

In [9]:
shout("hello")

입력 hello로 shout가 호출되었습니다


'HELLO'

In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

In [ ]:
# share=True를 추가하면 외부에 공개적으로 접근할 수 있게 됩니다
# 다음 주에 다룰 HuggingFace의 Spaces라는 플랫폼을 사용하면 더 영구적인 호스팅이 가능합니다
# 참고: 일부 백신 소프트웨어와 회사 방화벽은 share=True 사용을 막을 수 있습니다.
# 회사 네트워크에서 작업 중이라면 이 테스트는 건너뛰는 것을 권장합니다.

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True)

In [ ]:
# inbrowser=True를 추가하면 새 브라우저 창이 자동으로 열립니다

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

## 인증 추가하기

Gradio는 사용자 ID와 비밀번호를 매우 쉽게 사용할 수 있게 해줍니다

In [ ]:

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=("brunosong", "bananas"))

In [ ]:
# 다크 모드 강제 적용

# 이 변수를 정의한 뒤 Interface를 생성할 때 js=force_dark_mode를 전달하세요

force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never", js=force_dark_mode).launch()

In [ ]:
# 조금 더 추가해봅니다:

message_input = gr.Textbox(label="메시지를 입력하세요:", info="외칠 메시지를 입력하세요", lines=7)
message_output = gr.Textbox(label="응답:", lines=8)

view = gr.Interface(
    fn=shout,
    title="Shout",
    inputs=[message_input],
    outputs=[message_output],
    examples=["hello", "howdy"],
    flagging_mode="never"
    )
view.launch()

In [19]:
# 이제 함수를 "shout"에서 "message_gemini"로 바꿔봅시다

message_input = gr.Textbox(label="메시지를 입력하세요:", info="재미나이에게 보낼 메시지를 입력하세요", lines=7)
message_output = gr.Textbox(label="응답:", lines=8)

view = gr.Interface(
    fn=message_gemini,
    title="GEMINI",
    inputs=[message_input],
    outputs=[message_output],
    examples=["hello", "howdy"],
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


In [20]:
# 이제 Markdown을 사용해봅시다
# 아래 코드에서 참조되지도 않는데 system_message를 설정하는 게 왜 차이를 만드는지 궁금하신가요?
# system_message가 전역 변수라는 점을 이용하고 있는 것입니다 - 위의 message_gpt 함수에서 사용됩니다 (가서 확인해보세요)
# 훌륭한 소프트웨어 엔지니어링 관행은 아니지만, Jupyter Lab에서 R&D를 할 때는 꽤 흔한 방식입니다!

system_message = "당신은 코드 블록 없이 마크다운으로 응답하는 도움이 되는 어시스턴트입니다"

message_input = gr.Textbox(label="메시지를 입력하세요:", info="재미나이에게 보낼 메시지를 입력하세요", lines=7)
message_output = gr.Markdown(label="응답:")

view = gr.Interface(
    fn=message_gemini,
    title="GEMINI",
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        "트랜스포머 아키텍처를 일반인에게 설명해줘",
        "트랜스포머 아키텍처를 AI 엔지니어를 꿈꾸는 사람에게 설명해줘",
        ],
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


In [22]:
# 결과를 스트리밍으로 받아오는 호출을 만들어봅시다

def stream_gemini(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = gemini.chat.completions.create(
        model='gemini-3.1-flash-lite',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [23]:
message_input = gr.Textbox(label="메시지를 입력하세요:", info="GEMINI에게 보낼 메시지를 입력하세요", lines=7)
message_output = gr.Markdown(label="응답:")

view = gr.Interface(
    fn=stream_gemini,
    title="GEMINI",
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        "트랜스포머 아키텍처를 일반인에게 설명해줘",
        "트랜스포머 아키텍처를 AI 엔지니어를 꿈꾸는 사람에게 설명해줘",
        ],
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


# 회사 브로셔 생성기 만들기

In [ ]:
from scraper import fetch_website_contents

In [ ]:

# 이 역시 전형적인 실험적 사고방식입니다 - 위에서 사용한 전역 변수를 변경하고 있습니다:

system_message = """
당신은 회사 웹사이트 랜딩 페이지의 내용을 분석해
잠재 고객, 투자자, 채용 후보자를 위한 짧은 회사 브로셔를 작성하는 어시스턴트입니다.
코드 블록 없이 마크다운으로 응답하세요.
"""

In [ ]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"{company_name}에 대한 회사 브로셔를 생성해주세요. 다음은 그들의 랜딩 페이지입니다:\n"
    prompt += fetch_website_contents(url)
    if model=="GPT":
        result = stream_gemini(prompt)
    elif model=="Claude":
        result = stream_claude(prompt)  # 만들어야 함
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
name_input = gr.Textbox(label="회사 이름:")
url_input = gr.Textbox(label="랜딩 페이지 URL (http:// 또는 https:// 포함)")
model_selector = gr.Dropdown(["GPT", "Claude"], label="모델을 선택하세요", value="GPT")
message_output = gr.Markdown(label="응답:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator",
    inputs=[name_input, url_input, model_selector],
    outputs=[message_output],
    examples=[
            ["Hugging Face", "https://huggingface.co", "GPT"]
        ],
    flagging_mode="never"
    )
view.launch()